# Implementing Vector Search

Vector search finds similar items by comparing numerical representations (embeddings) of data. Here's a practical guide:

## Core Concepts

```
Text/Image → Embedding Model → Vector [0.2, 0.8, 0.1, ...] → Store & Search
```

$$\text{Cosine Similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$

**Similarity metrics:**
- **Cosine similarity** - angle between vectors (most common)
- **Euclidean distance** - straight-line distance
- **Dot product** - magnitude + direction


In [1]:
# =============================================================================
# Simple Vector Search (from scratch) using sentence-transformers + NumPy.
# -----------------------------------------------------------------------------
# This cell demonstrates the bare-bones mechanics of a semantic search engine:
#   1. Encode each document into a fixed-length embedding vector.
#   2. Encode the user query into the same embedding space.
#   3. Score every document against the query with cosine similarity.
#   4. Return the top-k highest-scoring documents.
# It is intentionally not optimized (no FAISS, no batching, no persistence)
# so the math is easy to follow.
# =============================================================================

import numpy as np
from sentence_transformers import SentenceTransformer


class SimpleVectorSearch:
    """Minimal in-memory vector search over a list of short documents."""

    def __init__(self):
        # 'all-MiniLM-L6-v2' is a lightweight 384-dim sentence embedding model
        # that's fast on CPU and good enough for demos / small corpora.
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.vectors = []     # parallel list of embedding vectors
        self.documents = []   # parallel list of original document strings

    def add_documents(self, docs: list[str]):
        """Encode each document into a vector and remember both."""
        self.documents.extend(docs)
        # .encode() returns a NumPy array shaped (n_docs, embedding_dim).
        embeddings = self.model.encode(docs)
        self.vectors.extend(embeddings)

    def search(self, query: str, top_k: int = 5) -> list[dict]:
        """Return the top_k documents most similar to `query`."""
        # Encode the query in the SAME embedding space as the documents.
        query_vector = self.model.encode([query])[0]

        # Stack the stored doc vectors into a matrix of shape (n_docs, dim)
        # so we can compute all similarities in one vectorized op.
        vectors_matrix = np.array(self.vectors)

        # Cosine similarity = (A . B) / (||A|| * ||B||)
        # Numerator: dot product of every doc vector with the query vector.
        # Denominator: product of L2 norms (per-doc norm * query norm).
        similarities = np.dot(vectors_matrix, query_vector) / (
            np.linalg.norm(vectors_matrix, axis=1) * np.linalg.norm(query_vector)
        )

        # argsort gives ascending order; [::-1] reverses to descending;
        # [:top_k] keeps only the best matches.
        top_indices = np.argsort(similarities)[::-1][:top_k]

        # Return a list of {document, score} dicts, ordered by relevance.
        return [
            {"document": self.documents[i], "score": float(similarities[i])}
            for i in top_indices
        ]


# -----------------------------------------------------------------------------
# Quick sanity check: index four short sentences and run three queries.
# Each query intentionally probes a different topic (AI, pets, sports) so
# we can eyeball whether the ranking matches our intuition.
# -----------------------------------------------------------------------------
search = SimpleVectorSearch()
search.add_documents([
    "Python is a programming language",
    "Machine learning uses neural networks",
    "Vector search finds similar content",
    "Dogs are popular pets",
])

# Query 1: AI/ML topic — expect the "machine learning" sentence on top.
results = search.search("How does AI work?")
for r in results:
    print(f"{r['score']:.3f} | {r['document']}")

print('##########################################')

# Query 2: Pet/animal topic — expect the "dogs" sentence on top.
results = search.search("Where can I find puppy?")
for r in results:
    print(f"{r['score']:.3f} | {r['document']}")

print('##########################################')

# Query 3: Cricket — no document is really about cricket, so scores
# should all be relatively low (a useful negative example).
results = search.search("Is the game of cricket the most popular?")
for r in results:
    print(f"{r['score']:.3f} | {r['document']}")


/opt/anaconda3/envs/AI-Python/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9412.26it/s]


0.484 | Machine learning uses neural networks
0.235 | Python is a programming language
0.157 | Vector search finds similar content
0.036 | Dogs are popular pets
##########################################
0.477 | Dogs are popular pets
0.104 | Vector search finds similar content
0.050 | Python is a programming language
0.050 | Machine learning uses neural networks
##########################################
0.344 | Dogs are popular pets
0.125 | Python is a programming language
0.077 | Machine learning uses neural networks
-0.006 | Vector search finds similar content


In [ ]:
# Utility: wipe the main Chroma collection used below.
# Uncomment and run only when you want to rebuild the index from the PDF.
# vector_db.delete_collection()


In [ ]:
# =============================================================================
# Build a persistent Chroma vector store from a PDF using HuggingFace embeddings.
# -----------------------------------------------------------------------------
# Pipeline:
#   PDF -> page-level Documents -> ~1000-char chunks -> 768-dim embeddings
#       -> persisted Chroma collection on disk.
# We use the higher-quality `all-mpnet-base-v2` model (768 dims) rather than
# the smaller MiniLM (384 dims) because retrieval quality matters more here
# than speed.
# =============================================================================

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# -----------------------------------------------------------------------------
# 1. Configure the embedding model.
#    - all-mpnet-base-v2 produces 768-dim vectors with strong semantic quality.
#    - MiniLM is left commented as a faster, lower-quality alternative.
#    - device='cpu' is portable; switch to 'cuda' if a GPU is available.
#    - normalize_embeddings=False keeps raw vectors; Chroma will use cosine
#      distance internally, so explicit normalization isn't required here.
# -----------------------------------------------------------------------------
model_name = "sentence-transformers/all-mpnet-base-v2"
# model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cpu'}            # Use 'cuda' if an NVIDIA GPU is available
encode_kwargs = {'normalize_embeddings': False}

hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

# -----------------------------------------------------------------------------
# 2. Load and chunk the source PDF.
#    PyPDFLoader returns one Document per page; the splitter then breaks those
#    pages into overlapping ~1000-char windows so a single chunk fits comfortably
#    in the embedding model's context and retrieval is fine-grained.
#    chunk_overlap=150 keeps continuity across chunk boundaries so a sentence
#    that straddles two chunks is still retrievable.
# -----------------------------------------------------------------------------
loader = PyPDFLoader("/Users/rajeevkumar/Downloads/" + "Employee Handbook - US.pdf")
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

# -----------------------------------------------------------------------------
# 3. Embed every chunk and persist them in a local Chroma collection.
#    `persist_directory` makes the index survive across notebook restarts so
#    we don't have to re-embed on every run.
# -----------------------------------------------------------------------------
vector_db = Chroma.from_documents(
    documents=docs,
    embedding=hf_embeddings,
    persist_directory="./chroma_mpnet_db",
    collection_name="employee_handbook_mpnet"
)

print(f"Finished loading {len(docs)} chunks using all-mpnet-base-v2.")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8424.68it/s]


Finished loading 538 chunks using all-mpnet-base-v2.


In [25]:
# =============================================================================
# Inspect what Chroma actually stored: original text + embedding vectors.
# Useful for sanity-checking chunk size, page boundaries, and vector shape.
# =============================================================================
import chromadb

# 1. Initialize the Chroma Client and connect to the collection.
client = chromadb.PersistentClient(path="./chroma_mpnet_db")

# # 2. Get a list of all collection objects
collections = client.list_collections()
print(f"Collections in Chroma DB: {[col.name for col in collections]}")

# # 3. Access the specific collection where we stored the chunks and embeddings.
collection = client.get_collection(name="employee_handbook_mpnet")

# 4. Pull EVERYTHING out of the collection. By default Chroma returns ids only;
# we have to explicitly ask for documents / metadata / embeddings.
results = collection.get(
    include=["documents", "metadatas", "embeddings"],
)

print(f"Total Documents Chunks Created : {len(results['ids'])}")

# Look at a small window of chunks. Adjust start/end to inspect other regions.
start = 60
end = 62

# We iterate by index (not by dict key) so we can line up
# documents <-> embeddings <-> metadatas by position.
for i in range(start, end):
    print(f"\n--- Index {i} ---")

    # Chroma can return None for fields that weren't stored; the explicit
    # `is not None` check avoids accidentally truth-testing a NumPy array
    # (which raises ValueError on ambiguous truthiness).
    if results['documents'] is not None:
        print(f"Original Document: {results['documents'][i]}")

    if results['embeddings'] is not None:
        import numpy as np
        # Wrap in np.array so we can call .shape regardless of whether
        # Chroma handed us a list or already a NumPy array.
        val = np.array(results['embeddings'][i])
        print(f"Rows and Dimensions : {val.shape}")          # expect (768,)
        print(f"Embedded Vector (First 10) : {val[:10]}")    # peek at the values


Collections in Chroma DB: ['employee_handbook_mpnet']
Total Documents Chunks Created : 538

--- Index 60 ---
Original Document: AIG Employee Handbook | June 16, 2025 | 23
CONFIDENTIAL – INTERNAL USE ONLY
Working Together (continued)
Welcome to AIG
Table of Contents
Working Together
Work-Life Balance
Your Career
Leaving the Company
Our Facilities
Information Technology,  
Intellectual Property  
and Media
Your Benefits
Appendix
Hours of Work and Overtime
It is AIG’s policy to pay for all time worked in accordance with 
applicable law. The following policies meet or exceed federal 
regulations. For specific information concerning this policy, 
including questions regarding recording work time, please 
contact HR Shared Services at 1-800-265-5054. 
Deviations from this policy present regulatory risk and must 
be pre-approved. If you feel a deviation is required, please 
contact your Compensation Partner.
Applicability
Most non-exempt employees who work a regular workweek 
are paid for tim

In [ ]:
# Scratch cell: uncomment to dump the full `results` dict in the notebook.
# Warning: with ~1k chunks this prints a *lot*.
# print(results)

In [28]:
# =============================================================================
# Plain similarity search (no LLM yet): ask the vector DB for the top-k chunks
# closest to a natural-language query. This is the "Retrieval" half of RAG.
# =============================================================================

vector_db_read = Chroma(
    embedding_function=hf_embeddings,
    persist_directory="./chroma_mpnet_db",
    collection_name="employee_handbook_mpnet"
)
query = "Tell me about paternity policy?"

# k=5 -> return the five most relevant chunks. Larger k = more context for the
# downstream LLM but more noise and more tokens.
results = vector_db_read.similarity_search(query, k=5)

# Print each retrieved chunk on its own block so we can eyeball whether the
# retriever is finding the right section of the handbook.
for doc in results:
    print(f"Result: {doc.page_content}\n")


Result: which purports to deny an employee the right to make 
their own reproductive health care decisions, including use 
of a particular drug, device, or medical service. 
An employee may bring a civil action in any court of 
competent jurisdiction against an employer for any alleged 
violations of this policy. In any civil action alleging a violation 
of this policy, the court may: award damages, including, 
but not limited to, back pay, benefits and reasonable 
attorneys’ fees and costs incurred to a prevailing plaintiff; 
afford injunctive relief against the employer if it commits or 
proposes to commit a violation of the provisions of this policy; 
order reinstatement; and/or award liquidated damages equal 
to 100% of the award for damages unless the employer proves 
a good faith basis to believe that its actions in violation of this 
policy were in compliance with the law. 
Any act of retaliation against an employee for exercising any

Result: worked at least 1,250 hours during 

In [30]:
# =============================================================================
# Second retrieval probe — same mechanism, different topic.
# Comparing the two queries side-by-side is a quick way to gauge whether the
# embedding model can distinguish unrelated policy sections.
# =============================================================================

query = "Tell me about 401-k policy?"
results = vector_db_read.similarity_search(query, k=5)

for doc in results:
    print(f"Result: {doc.page_content}\n")


Result: employment, nor a legally binding agreement and does not 
create a contract for wages or any other working conditions. 
AIG may, in its sole discretion, alter or amend these policies 
at any time, with or without notice. You are responsible for 
being aware of, and abiding by, all of the Handbook’s policies, 
including any of its future modifications and understand that 
the current version of the Handbook is available for review at 
any time on AIG’s intranet.
This Handbook Does Not Modify AIG’s  
Benefit Plans
The Handbook includes some information about employee 
benefits that may appear in other AIG employee materials 
such as summary plan descriptions or plan documents. The 
descriptions of policies and benefit plans in this Handbook 
are not intended as substitutes for the documents that 
legally govern each policy and benefit plan. If there is any 
conflict between the employee benefits programs described 
in this Handbook and the information contained in other AIG

Resu

In [32]:
# =============================================================================
# Full RAG (Retrieval-Augmented Generation) pipeline:
#   1. Use the vector DB to fetch chunks relevant to the user's question.
#   2. Concatenate those chunks into a context block.
#   3. Send {system prompt + context + question} to Claude.
#   4. Print Claude's grounded answer.
# This is the pattern that lets the model answer questions about a private
# document (the AIG Employee Handbook) it was never trained on.
# =============================================================================

import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# -----------------------------------------------------------------------------
# 1. Retrieval — pull the top-k most relevant chunks from Chroma.
#    Alternate queries are kept above as quick A/B test toggles.
# -----------------------------------------------------------------------------
# query = "Tell me about paternity policy?"
# query = "Tell me about leaves entitlement?"
# query = "Tell me about unapproved leave policy and its implications of employment?"
query = "Tell me about work attire policy?"

# k=10 gives the LLM a fairly wide context window over the handbook.
# Tune k up for recall-heavy questions, down to reduce token cost / noise.
docs = vector_db.similarity_search(query, k=10)

# -----------------------------------------------------------------------------
# 2. Stitch retrieved chunks together into a single context string.
#    A blank line between chunks helps the model treat them as separate passages.
# -----------------------------------------------------------------------------
context = "\n\n".join([doc.page_content for doc in docs])

# -----------------------------------------------------------------------------
# 3. Build the prompt.
#    - The system prompt sets the role and tells the model to ground answers
#      in the provided context, and to admit when the answer isn't present.
#    - The user message bundles the retrieved context with the question.
# -----------------------------------------------------------------------------
system_prompt = """You are an expert data assistant. Use the provided context
to answer the user's question. If the answer isn't in the context, say so."""

user_message = f"""Context:
{context}

Question: {query}"""

# -----------------------------------------------------------------------------
# 4. Generation — send the grounded prompt to Claude and print the answer.
# -----------------------------------------------------------------------------
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system=system_prompt,
    messages=[
        {"role": "user", "content": user_message}
    ],
)

print(response.content[0].text)


# AIG Work Attire Policy (Dress for Your Day)

## Overview
AIG has a **"Dress for Your Day"** dress code designed to help employees feel more comfortable in the workplace while maintaining professionalism. Appearance must always be **appropriate for the circumstances**, including video conferencing and client meetings.

---

## Unacceptable Clothing Items
The following are generally **not permitted** at work:
- Jeans with rips and holes
- Overly tight or revealing attire
- Slippers, sneakers, or flip flops
- Hats or shorts
- Gym attire
- Stained, wrinkled, frayed, or torn clothing
- Clothing with suggestive or inappropriate statements

---

## Key Guidelines
- **Traditional business attire** may be required when meeting clients or representing the Company at outside functions
- Some **business units may have different dress requirements** — managers will advise accordingly
- Managers have the authority to deem clothing inappropriate and may **send employees home**, subject to Correctiv